In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import jax
import jax.numpy as jnp
import random
import numpy as np
from einops import rearrange
from jax import jit, vmap
from hoam.plot import scatter_movie, imshow_movie
import matplotlib.pyplot as plt

In [2]:
from hoam.vlasov import run_vlasov

n_samples = 10_000 
dt = 1e-2
t_end = 40
t_eval = np.linspace(0.0, t_end, int(t_end/dt)+1)
train_mus = np.asarray([1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9])
test_mus = np.asarray([1.25, 1.85])
mus = np.concatenate([train_mus, test_mus])
train_sols = []
for mu in train_mus:
    res = run_vlasov(n_samples, t_eval, mu, mode='two-stream')
    train_sols.append(res)
    
test_sols = []
for mu in test_mus:
    res = run_vlasov(n_samples, t_eval, mu, mode='two-stream')
    train_sols.append(res)
    
train_sols = np.asarray(train_sols)
test_sols = np.asarray(test_sols)

NameError: name 'np' is not defined

In [4]:
from hoam.sde import solve_sde

sol = solve_sde(drift, diffusion, t_eval, get_inital_condition, n_samples, dt=1e-3, key=key)
sol = sol[..., :2]
sol = rearrange(sol, 'N T D -> T N D')

In [5]:
scatter_movie(sol)

In [6]:
from hoam.colora import build_colora

key = jax.random.PRNGKey(1)

x_dim = 2
mu_t_dim = 2
u_dim = 1

u_hat_config = {'width': 25, 'layers': ['C']*7}
h_config = {'width': 15, 'layers': ['D']*3}

s_fn_raw, params_init = build_colora(u_hat_config, h_config, x_dim, mu_t_dim, u_dim, rank=3, key=key)

In [7]:
def E_rho(t):
    mu_0 = jnp.asarray([1, 1])*-1
    return mu_0*jnp.cos(omega*t)

def s_fn(mu, X_batch, t_batch, params):
    x_pt = E_rho(jnp.squeeze(t_batch))
    return s_fn_raw(mu, X_batch, t_batch, params) - s_fn_raw(mu, x_pt, t_batch, params)
    

In [8]:
from hoam.utils import get_rand_idx, interplate_in_t
from hoam.quad import get_simpsons

# here we prepare the data for quadtrature

bs_t = 256 # batch size in time
bs_n = 256 # batch size in samples

# necessary for simpsons
if (bs_t - 1) % 2 != 0:
    bs_t += 1
    
t_batch, quad_weights = get_simpsons(bs_t)
start, end = jnp.asarray([0]), jnp.asarray([1.0])
t_batch = jnp.concatenate([start, t_batch, end])
t_batch = t_batch.reshape(-1, 1)
        
X_batch = interplate_in_t(sol, t_eval, t_batch)
mu_batch = jnp.asarray([1.0]) # dummy mu for this example
    
print(X_batch.shape, t_batch.shape)

(259, 10000, 2) (259, 1)


In [9]:

def get_sample_fn(X_batch, t_batch, mu_batch):
    X_batch = jnp.asarray(X_batch)
    t_batch = jnp.asarray(t_batch)
    mu_batch = jnp.asarray(mu_batch)
    def sample_fn(in_key):

        nonlocal X_batch
        nonlocal t_batch
        nonlocal mu_batch
    
        T, N, D = X_batch.shape
        T, one = t_batch.shape
        
        keys = jax.random.split(in_key, num=T)
        sample_idx = vmap(get_rand_idx, (0, None, None))(keys, N, bs_n)

        rows = jnp.arange(X_batch.shape[0])[:, jnp.newaxis]
        X_batch = X_batch[rows, sample_idx]

        return mu_batch, X_batch, t_batch, quad_weights

    return sample_fn

sample_fn = get_sample_fn(X_batch, t_batch, mu_batch)
sample_fn = jit(sample_fn)

In [10]:
from hoam.utils import meanvmap, tracewrap
from jax import jacrev, jacfwd

s_Ex = meanvmap(s_fn, in_axes=(None, 0, None, None))
s_Ex_Vt = vmap(s_Ex, in_axes=(None, 0, 0, None))

s_dx = jacrev(s_fn, 1)
s_dx_norm = lambda *args: jnp.sum(s_dx(*args)**2)
s_dx_norm_Ex = meanvmap(s_dx_norm, in_axes=(None, 0, None, None))
s_dx_norm_Ex_Vt = vmap(s_dx_norm_Ex, in_axes=(None, 0, 0, None))

s_dt = jacrev(s_fn, 2)
dt_Ex = meanvmap(s_dt, in_axes=(None, 0, None, None))
dt_Ex_Vt = vmap(dt_Ex, in_axes=(None, 0, 0, None))

laplace = tracewrap(jacfwd(s_dx, 1))
laplace_norm = lambda *args: jnp.sum(laplace(*args)**2)
laplace_norm_Ex = meanvmap(laplace_norm, in_axes=(None, 0, None, None))
laplace_norm_Ex_Vt = vmap(laplace_norm_Ex, in_axes=(None, 0, 0, None))


epsilon = 5e-2

def loss_fn(params, mu, X_batch, t_batch, quad_weights):
    
    
    boundary_term = s_Ex(mu, X_batch[0], t_batch[0], params) - s_Ex(mu,  X_batch[-1], t_batch[-1], params)

    X_batch = X_batch[1:-1]
    t_batch = t_batch[1:-1]
    
    grad = s_dx_norm_Ex_Vt(mu, X_batch, t_batch, params)
    dt = dt_Ex_Vt(mu, X_batch, t_batch, params)
    dt = jnp.squeeze(dt)
    
    if epsilon > 0.0:
        lap = laplace_norm_Ex_Vt(mu, X_batch, t_batch, params)
    else:
        lap = 0.0
    
    interior = 0.5*grad + dt + epsilon**2*0.5*lap

    interior_loss = (interior * quad_weights).sum()
    
    loss = interior_loss + boundary_term
    
    return loss

In [12]:
from hoam.adam import adam_opt

opt_params, loss_history = adam_opt(params_init, loss_fn, sample_fn, steps=10_000, learning_rate=2e-3, verbose=True, key=key)


  0%|          | 0/10000 [00:00<?, ?it/s]

In [13]:
from hoam.sde import solve_test_sde

ic = sol[0]
t_int =  jnp.linspace(0.0, 1.0, len(t_eval))
dt_test = 1e-3
test_sol = solve_test_sde(s_fn, opt_params, ic, t_int, dt_test, epsilon, mu_batch, key)


In [14]:
n_plot = 512
N = test_sol.shape[1]
idx_sample = np.linspace(0, N-1, n_plot, dtype=np.uint32)
cs = [*['r']*n_plot, *['b']*n_plot]
plot_sol = jnp.hstack([sol[:, idx_sample], test_sol[:, idx_sample]])
scatter_movie(plot_sol, c=cs, alpha=0.2)